# Imports

In [1]:
from pathlib import Path
import numpy as np
import numpy.typing as npt
import pandas as pd
import sklearn
from sklearn.metrics.pairwise import cosine_similarity
import random
import matplotlib.pyplot as plt
import seaborn as sns

# Data loading and Splitting

In [2]:
interactions = pd.read_csv(Path.cwd().parent/"data"/"interactions_train.csv")
items = pd.read_csv(Path.cwd().parent/"data"/"items.csv")

In [14]:
interactions

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09
...,...,...,...
87042,924,8171,1.699284e+09
87043,1106,9009,1.699872e+09
87044,5207,13400,1.683627e+09
87045,698,5779,1.686667e+09


In [6]:
unique_items = interactions.i.unique().tolist()
unique_items.sort()

id_in_order = [unique_items.index(i) for i in interactions["i"]]
interactions["i in order"] = id_in_order

interactions

,u,i,t,i in order
0,4456,8581,1.687541e+09,8509
1,142,1964,1.679585e+09,1944
2,362,3705,1.706872e+09,3677
3,1809,11317,1.673533e+09,11216
4,4384,1323,1.681402e+09,1306
...,...,...,...,...
87042,924,8171,1.699284e+09,8103
87043,1106,9009,1.699872e+09,8931
87044,5207,13400,1.683627e+09,13282
87045,698,5779,1.686667e+09,5735


In [3]:
n_users = interactions["u"].nunique()
n_items = interactions["i"].nunique()
print(f"Number of users: {n_users}, number of items: {n_items}, number of interactions: {len(interactions)}")

Number of users: 7838, number of items: 15109, number of interactions: 87047


In [22]:
ref = 0
inter_list = interactions["i"].unique().tolist()
print(f"Minimum item ID: {min(inter_list)}, Maximum item ID: {max(inter_list)}")
inter_list.sort()
for i in inter_list:
    if i != ref:
        print(f"Item {i} is NOT in the interactions dataset.")
        ref = i
    ref+=1

Minimum item ID: 0, Maximum item ID: 15290
Item 96 is NOT in the interactions dataset.
Item 211 is NOT in the interactions dataset.
Item 514 is NOT in the interactions dataset.
Item 516 is NOT in the interactions dataset.
Item 518 is NOT in the interactions dataset.
Item 522 is NOT in the interactions dataset.
Item 525 is NOT in the interactions dataset.
Item 648 is NOT in the interactions dataset.
Item 836 is NOT in the interactions dataset.
Item 841 is NOT in the interactions dataset.
Item 846 is NOT in the interactions dataset.
Item 851 is NOT in the interactions dataset.
Item 893 is NOT in the interactions dataset.
Item 1259 is NOT in the interactions dataset.
Item 1460 is NOT in the interactions dataset.
Item 1652 is NOT in the interactions dataset.
Item 1692 is NOT in the interactions dataset.
Item 2362 is NOT in the interactions dataset.
Item 2569 is NOT in the interactions dataset.
Item 2607 is NOT in the interactions dataset.
Item 2661 is NOT in the interactions dataset.
Item 

In [23]:
ref = 0
user_list = interactions["u"].unique().tolist()
print(f"Minimum user ID: {min(user_list)}, Maximum user ID: {max(user_list)}")
user_list.sort()
for i in user_list:
    if i != ref:
        print(f"User {i} is NOT in the interactions dataset.")
        ref = i
    ref+=1

Minimum user ID: 0, Maximum user ID: 7837


In [24]:
interactions.i.max()

np.int64(15290)

In [4]:
interactions = interactions.sort_values(["u", "t"])
interactions

,u,i,t
21035,0,0,1.680191e+09
28842,0,1,1.680783e+09
3958,0,2,1.680801e+09
29592,0,3,1.683715e+09
6371,0,3,1.683715e+09
...,...,...,...
74604,7836,3471,1.728644e+09
69092,7836,3471,1.728644e+09
51684,7837,2191,1.728735e+09
76172,7837,88,1.728735e+09


In [5]:
interactions["pct_rank"] = interactions.groupby("u")["t"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)
interactions

,u,i,t,pct_rank
0,0,0,1.680191e+09,0.040000
1,0,1,1.680783e+09,0.080000
2,0,2,1.680801e+09,0.120000
3,0,3,1.683715e+09,0.160000
4,0,3,1.683715e+09,0.200000
...,...,...,...,...
87042,7836,3471,1.728644e+09,0.666667
87043,7836,3471,1.728644e+09,1.000000
87044,7837,2191,1.728735e+09,0.333333
87045,7837,88,1.728735e+09,0.666667


In [6]:
train_data = interactions[interactions["pct_rank"] < 0.8]
test_data = interactions[interactions["pct_rank"] >= 0.8]

print("Training set size:", train_data.shape[0])
print("Testing set size:", test_data.shape[0])

Training set size: 65419
Testing set size: 21628


# User-Item Matrices

In [8]:
interactions.u.unique()

array([   0,    1,    2, ..., 7835, 7836, 7837], shape=(7838,))

In [10]:
def create_data_mtx(data : pd.DataFrame, n_users: int, n_items: int) -> npt.NDArray[np.float64]:
    """
    Create a user-item interaction matrix from the given data.

    :param data: A DataFrame containing user-item interactions with columns "u" for user IDs and "i" for item IDs.
    :type data: pd.DataFrame
    :param n_users: The number of users.
    :type n_users: int
    :param n_items: The number of items.
    :type n_items: int
    :return: A user-item interaction matrix.
    :rtype: npt.NDArray[np.float64]
    """
    # data_mtx = np.zeros((n_users, n_items))    # Ensure the matrix can accommodate all item IDs
    # data_mtx[data["u"].values, data["i in order"].values] = 1
    data_mtx = np.zeros((n_users, max(interactions.i.unique())+1))
    data_mtx[data["u"].values, data["i"].values] = 1
    return data_mtx

train_data_mtx = create_data_mtx(train_data, n_users, n_items)
test_data_mtx = create_data_mtx(test_data, n_users, n_items)
print("Number of interactions in training set:", np.sum(train_data_mtx))
print("Number of interactions in test set:", np.sum(test_data_mtx))

Number of interactions in training set: 49689.0
Number of interactions in test set: 19409.0


# Item-to-Item

In [11]:
item_similarity = cosine_similarity(train_data_mtx.T)
item_similarity.shape

(15291, 15291)

In [12]:
def item_based_predict(interactions : pd.DataFrame, similarity : npt.NDArray[np.float64], epsilon=1e-9):
    """
    Predicts user-item interactions based on item-item similarity.

    :param interactions: The user-item interaction matrix.
    :type interactions: pd.DataFrame
    :param similarity: The item-item similarity matrix.
    :type similarity: npt.NDArray[np.float64]
    :param epsilon: Small constant added to the denominator to avoid division by zero.
    :type epsilon: float
    :return: The predicted interaction scores for each user-item pair.
    :rtype: npt.NDArray[np.float64]
    """
    # np.dot does the matrix multiplication. Here we are calculating the
    # weighted sum of interactions based on item similarity
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T  # Transpose to get users as rows and items as columns

item_prediction = item_based_predict(train_data_mtx, item_similarity)
item_prediction.shape

(7838, 15291)

# User-to-User

In [13]:
user_similarity = cosine_similarity(train_data_mtx)
user_similarity.shape

(7838, 7838)

In [14]:
def user_based_predict(interactions, similarity, epsilon=1e-9):
    """
    Predicts user-item interactions based on user-user similarity.
    
    :param interactions: The user-item interaction matrix.
    :type interactions: npt.NDArray[np.float64]
    :param similarity: The user-user similarity matrix.
    :type similarity: npt.NDArray[np.float64]
    :param epsilon: Small constant added to the denominator to avoid division by zero.
    :type epsilon: float
    :return: The predicted interaction scores for each user-item pair.
    :rtype: npt.NDArray[np.float64]
    """
    # Calculate the weighted sum of interactions based on user similarity
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

# Calculate the user-based predictions for positive interactions
user_prediction = user_based_predict(train_data_mtx, user_similarity)
user_prediction.shape

(7838, 15291)

# Evaluation

In [15]:
def precision_recall_at_k(prediction, ground_truth, k=10):
    """
    Calculates Precision@K and Recall@K for top-K recommendations.

    :param prediction: The predicted interaction matrix with scores.
    :type prediction: npt.NDArray[np.float64]
    :param ground_truth: The ground truth interaction matrix (binary).
    :type ground_truth: npt.NDArray[np.float64]
    :param k: Number of top recommendations to consider.
    :type k: int
    :return: The average precision@K and recall@K over all users.
    :rtype: tuple[float, float]
    """
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        top_k_items = np.argsort(prediction[user, :])[-k:]
        relevant_items_in_top_k = np.isin(top_k_items, np.where(ground_truth[user, :] == 1)[0]).sum()
        total_relevant_items = ground_truth[user, :].sum()
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0
    precision_at_k /= num_users
    recall_at_k /= num_users
    
    return precision_at_k, recall_at_k

precision_user_k, recall_user_k = precision_recall_at_k(user_prediction, test_data_mtx, k=10)
precision_item_k, recall_item_k = precision_recall_at_k(item_prediction, test_data_mtx, k=10)

print('User-based CF Precision@K:', precision_user_k)
print('User-based CF Recall@K:', recall_user_k)
print('Item-based CF Precision@K:', precision_item_k)
print('Item-based CF Recall@K:', recall_item_k)

User-based CF Precision@K: 0.05651952028578994
User-based CF Recall@K: 0.2904556034557242
Item-based CF Precision@K: 0.05572850216892345
Item-based CF Recall@K: 0.26418648020233576


In [11]:
def precision_recall_at_k(prediction, ground_truth, k=10):
    """
    Calculates Precision@K and Recall@K for top-K recommendations.

    :param prediction: The predicted interaction matrix with scores.
    :type prediction: npt.NDArray[np.float64]
    :param ground_truth: The ground truth interaction matrix (binary).
    :type ground_truth: npt.NDArray[np.float64]
    :param k: Number of top recommendations to consider.
    :type k: int
    :return: The average precision@K and recall@K over all users.
    :rtype: tuple[float, float]
    """
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        top_k_items = np.argsort(prediction[user, :])[-k:]
        relevant_items_in_top_k = np.isin(top_k_items, np.where(ground_truth[user, :] == 1)[0]).sum()
        total_relevant_items = ground_truth[user, :].sum()
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0
    precision_at_k /= num_users
    recall_at_k /= num_users
    
    return precision_at_k, recall_at_k

precision_user_k, recall_user_k = precision_recall_at_k(user_prediction, test_data_mtx, k=10)
precision_item_k, recall_item_k = precision_recall_at_k(item_prediction, test_data_mtx, k=10)

print('User-based CF Precision@K:', precision_user_k)
print('User-based CF Recall@K:', recall_user_k)
print('Item-based CF Precision@K:', precision_item_k)
print('Item-based CF Recall@K:', recall_item_k)

User-based CF Precision@K: 0.056557795355960915
User-based CF Recall@K: 0.29074409391720035
Item-based CF Precision@K: 0.055702985455476146
Item-based CF Recall@K: 0.26416329763269525


#  Build submission

In [16]:
sample_submission = pd.read_csv(Path.cwd().parent/"data"/"sample_submission.csv")
submission_user = pd.DataFrame()
submission_user["user_id"] = sample_submission["user_id"]
submission_item = submission_user.copy()
predictions_user = []
precision_item = []
for u in submission_user["user_id"]:
    top_10_items = np.argsort(user_prediction[u, :])[-10:][::-1]
    top_10_users = np.argsort(item_prediction[u, :])[-10:][::-1]
    predictions_user.append(" ".join(map(str, top_10_users)))
    precision_item.append(" ".join(map(str, top_10_items)))
submission_user["recommendation"] = predictions_user
submission_item["recommendation"] = precision_item
submission_user.to_csv(Path.cwd().parent/"submissions"/"submission_user_v2.csv", index=False)
submission_item.to_csv(Path.cwd().parent/"submissions"/"submission_item_v2.csv", index=False)

In [20]:
full_data_mtx = create_data_mtx(interactions, n_users, n_items)
print("Number of interactions in full set:", np.sum(full_data_mtx))

# 2. Recompute Item-Item similarities and predictions on the full dataset
print("Computing full Item-based predictions...")
full_item_similarity = cosine_similarity(full_data_mtx.T)
full_item_prediction = item_based_predict(full_data_mtx, full_item_similarity)

# 3. Recompute User-User similarities and predictions on the full dataset
print("Computing full User-based predictions...")
full_user_similarity = cosine_similarity(full_data_mtx)
full_user_prediction = user_based_predict(full_data_mtx, full_user_similarity)

# 4. Build the final submission using the completely retrained predictions
sample_submission = pd.read_csv(Path.cwd().parent/"data"/"sample_submission.csv")

submission_user = pd.DataFrame()
submission_user["user_id"] = sample_submission["user_id"]

submission_item = pd.DataFrame()
submission_item["user_id"] = sample_submission["user_id"]

predictions_user = []
predictions_item = []

full_user_prediction

for u in submission_user["user_id"]:
    # We want to recommend items the user HAS NOT interacted with yet.
    # To do this cleanly, we can set the scores of already interacted items to a large negative number
    # (or simply 0 if your scores are strictly positive) before taking the top 10.
    
    # Copy the scores for the user
    u_scores_user_cf = full_user_prediction[u, :].copy()
    u_scores_item_cf = full_item_prediction[u, :].copy()
    
    # Zero out items the user has already interacted with (so we don't recommend a read book)
    # already_read_indices = np.where(full_data_mtx[u, :] > 0)[0]
    # u_scores_user_cf[already_read_indices] = -np.inf
    # u_scores_item_cf[already_read_indices] = -np.inf
    
    # Get top 10 items
    # top_10_users_cf = np.argsort(u_scores_user_cf)[-10:][::-1] # [::-1] ensures descending order
    # top_10_items_cf = np.argsort(u_scores_item_cf)[-10:][::-1]
    top_10_users_cf = np.argsort(u_scores_user_cf)[-10:]
    top_10_items_cf = np.argsort(u_scores_item_cf)[-10:]

    temp_u = []
    temp_i = []
    i_in_order = interactions["i in order"].to_list()
    for i in top_10_users_cf:
        temp_u.append(interactions["i"][i_in_order.index(i)])
    top_10_users_cf = np.array(temp_u)
    for i in top_10_items_cf:
        temp_i.append(interactions["i"][i_in_order.index(i)])
    top_10_items_cf = np.array(temp_i)
    
    predictions_user.append(" ".join(map(str, top_10_users_cf)))
    predictions_item.append(" ".join(map(str, top_10_items_cf)))

submission_user["recommendation"] = predictions_user
submission_item["recommendation"] = predictions_item

# Save files
submission_user.to_csv(Path.cwd().parent/"submissions"/"submission_user_full_unfiltered_v2.csv", index=False)
submission_item.to_csv(Path.cwd().parent/"submissions"/"submission_item_full_unfiltered_v2.csv", index=False)

Number of interactions in full set: 64003.0
Computing full Item-based predictions...
Computing full User-based predictions...
